# Consolidated Solar Transit Pipeline: Video to Diameter
### A Complete Astronomical Physics Workstation

This notebook contains the full end-to-end pipeline for your Solar Angular Diameter experiment:
1. **Video Extraction**: Invokes FFmpeg directly to extract raw uncompressed video frames into a dedicated folder.
2. **Drift Map Projection**: Generates a continuous maximum-intensity sweep map of the Earth's rotation pulling the Sun.
3. **Circle-Lock Interface**: Launches an interactive widget canvas. Pressing **`S`** freezes a blue geometric target ring around your initial position. Navigating forward with your **arrow keys** allows you to see the equatorial sunspot cross the diameter until you hit **`E`** at the exit edge.
4. **Orbital Derivation**: Computes the true transit time, the Astronomical Unit (AU), and the Sun's diameter in kilometers.

### Environment Setup
Ensure you have your environment ready:
```bash
pip install opencv-python numpy matplotlib ipywidgets scipy
```

## Step 1: Global Settings & Automated FFmpeg Extraction

In [1]:
!rm -rf extracted_solar_frames/

In [2]:
import os
import glob
import subprocess
import numpy as np
import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# =====================================================================
# 🛠️ PARAMETER CONFIGURATION
# =====================================================================
FRAME_OUTPUT_DIR = "extracted_solar_frames"
os.makedirs(FRAME_OUTPUT_DIR, exist_ok=True)
VIDEO_INPUT_FILE = "Solar_timelapse.mp4"
REAL_SECONDS_PER_FRAME = 1.0
# =====================================================================

os.makedirs(FRAME_OUTPUT_DIR, exist_ok=True)

print(f"Executing FFmpeg: Slicing video frames from '{VIDEO_INPUT_FILE}'...")
ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-i", VIDEO_INPUT_FILE,
    "-q:v", "2",
    os.path.join(FRAME_OUTPUT_DIR, "frame_%04d.jpg")
]

try:
    result = subprocess.run(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
    image_paths = sorted(glob.glob(os.path.join(FRAME_OUTPUT_DIR, "*.jpg")))
    print(f"🎉 Success! Extracted {len(image_paths)} frames into '{FRAME_OUTPUT_DIR}'")
except subprocess.CalledProcessError as e:
    print("❌ FFmpeg Extraction Failed. Error log:")
    print(e.stderr)

Executing FFmpeg: Slicing video frames from 'Solar_timelapse.mp4'...
🎉 Success! Extracted 410 frames into 'extracted_solar_frames'


## Step 2: Build the Continuous Sky Sweep Path

In [3]:
print("Generating maximum-intensity sweep map of the Earth's rotation track...")
first_img = cv2.imread(image_paths[0])
drift_composite = first_img.astype(np.float32)

for path in image_paths[1:]:
    img = cv2.imread(path)
    if img is None: continue
    drift_composite = np.maximum(drift_composite, img.astype(np.float32))

drift_composite_result = drift_composite.astype(np.uint8)
background_base = cv2.cvtColor(drift_composite_result, cv2.COLOR_BGR2RGB)
print("Background sweep map generation complete.")

Generating maximum-intensity sweep map of the Earth's rotation track...
Background sweep map generation complete.


## Step 3: Interactive Circle-Lock Interface & Orbital Calculations

In [4]:
state = {
    "start_idx": None,
    "end_idx": None,
    "frozen_circle": None
}

output_view = widgets.Output()
slider = widgets.IntSlider(
    value=0, min=0, max=len(image_paths) - 1, step=1,
    description='Frame Map:', layout=widgets.Layout(width='70%')
)

def find_solar_boundary(frame_idx):
    img = cv2.imread(image_paths[frame_idx])
    if img is None: return None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours: return None
    sun_contour = max(contours, key=cv2.contourArea)
    (x, y), r = cv2.minEnclosingCircle(sun_contour)
    return (int(x), int(y), int(r))

def calculate_astronomical_system():
    if state["start_idx"] is None or state["end_idx"] is None:
        return ""
    
    frame_delta = abs(state["end_idx"] - state["start_idx"])
    transit_seconds = frame_delta * REAL_SECONDS_PER_FRAME
    
    # 🌌 CLASSIC GEOMETRIC DERIVATION SYSTEM
    SECONDS_IN_DAY = 86400.0
    SPEED_OF_LIGHT_KM_S = 299792.458
    LIGHT_TRAVEL_TIME_SUNDIST_SECONDS = 499.005 
    
    # 1. Derive the True Astronomical Unit (AU) via Light Constant
    derived_au_km = SPEED_OF_LIGHT_KM_S * LIGHT_TRAVEL_TIME_SUNDIST_SECONDS
    textbook_au_km = 149597870.7
    au_error_margin = abs(derived_au_km - textbook_au_km) / textbook_au_km * 100
    
    # 2. Compute total space sweep circumference of Earth's path around Sun
    orbit_circumference_km = 2 * np.pi * derived_au_km
    
    # 3. Calculate true Linear Solar Diameter across the space transit vector fraction
    solar_diameter_km = (transit_seconds / SECONDS_IN_DAY) * orbit_circumference_km
    textbook_sun_km = 1392700.0
    sun_error_margin = abs(solar_diameter_km - textbook_sun_km) / textbook_sun_km * 100
    
    summary = (
        f"\n================= 🌌 FINAL DERIVED ASTRONOMICAL DATA =================\n"
        f"Start Target Frame: {state['start_idx']}  -->  End Target Frame: {state['end_idx']}\n"
        f"Total Segment Shift:          {frame_delta} frames\n"
        f"Measured Transit Duration:    {transit_seconds:.2f} seconds\n\n"
        #f"Derived Astronomical Unit(AU): {derived_au_km:,.1f} km\n"
        #f"Accepted Textbook AU Value:   {textbook_au_km:,.1f} km\n"
        #f"AU Experimental Margin Error: {au_error_margin:.4f}%\n\n"
        f"Derived Sun Linear Diameter:  {solar_diameter_km:,.1f} km\n"
        f"Accepted Textbook Sun Value:  {textbook_sun_km:,.1f} km\n"
        f"Sun Experimental Margin Error:{sun_error_margin:.2f}%\n"
        f"======================================================================"
    )
    return summary


def render_frame(frame_idx):
    with output_view:
        clear_output(wait=True)
        img = cv2.imread(image_paths[frame_idx])
        if img is None: return
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if state["frozen_circle"] is not None:
            cX, cY, radius = state["frozen_circle"]
            cv2.circle(img_rgb, (cX, cY), radius, (0, 150, 255), 4)
            
        blended_visualization = cv2.addWeighted(background_base, 0.4, img_rgb, 0.6, 0)
        
        plt.figure(figsize=(10, 10))
        plt.imshow(blended_visualization)
        
        title_text = f"Active Frame Index: {frame_idx}\n"
        title_text += f"[S] Lock Entrance: {state['start_idx'] if state['start_idx'] is not None else 'Not Set'} | "
        title_text += f"[E] Lock Exit: {state['end_idx'] if state['end_idx'] is not None else 'Not Set'}\n"
        if state["frozen_circle"]:
            title_text += "🔵 BOUNDARY LOCKED - Use arrow keys to track your sunspot to the opposite side of the ring"
            
        plt.title(title_text, fontsize=11, fontweight='bold')
        plt.axis("on")
        plt.show()
        
        if state["start_idx"] is not None and state["end_idx"] is not None:
            print(calculate_astronomical_system())

def handle_passthrough_hotkeys(change):
    key = change['new'].strip().lower()
    current_val = slider.value
    if key == 's':
        state["start_idx"] = current_val
        state["frozen_circle"] = find_solar_boundary(current_val)
        render_frame(current_val)
    elif key == 'e':
        state["end_idx"] = current_val
        render_frame(current_val)
    kbd_passport.value = ""

slider.observe(lambda change: render_frame(change['new']), names='value')
kbd_passport = widgets.Text(description="", layout=widgets.Layout(display='none'))
kbd_passport.observe(handle_passthrough_hotkeys, names='value')

js_hotkey_bridge = """
<script>
document.addEventListener('keydown', function(event) {
    var inputWidget = document.querySelector('.widget-text input');
    if (!inputWidget) return;
    if (event.key.toLowerCase() === 's') {
        inputWidget.value = 's';
        inputWidget.dispatchEvent(new Event('change', { bubbles: true }));
    } else if (event.key.toLowerCase() === 'e') {
        inputWidget.value = 'e';
        inputWidget.dispatchEvent(new Event('change', { bubbles: true }));
    }
});
</script>
"""

print("\n--- 📡 PIPELINE ACTIVE: INTERACTIVE VIEW STATION MODULE ---")
display(slider, kbd_passport, output_view)
display(HTML(js_hotkey_bridge))
render_frame(0)


--- 📡 PIPELINE ACTIVE: INTERACTIVE VIEW STATION MODULE ---


IntSlider(value=0, description='Frame Map:', layout=Layout(width='70%'), max=409)

Text(value='', layout=Layout(display='none'))

Output()